# Brain Tumor Segmentation — Evaluation & Inference

Loads the trained `best_model.pth` (mean Dice 0.8490 from the completed training run) and runs:
1. Full test-set evaluation (Dice/IoU/HD95 + comparison figures)
2. Single-case inference with a GT-vs-prediction visualization

**Before running:**
1. **Add Data -> Upload** your local `best_model.pth` (from `kaggle_output_v2/checkpoints/best_model.pth` on your computer) as a new private Kaggle Dataset, and attach it here.
2. **Add Data** -> search `brats20-dataset-training-validation` -> Add.
3. **Settings -> Accelerator** -> GPU T4 x2 (evaluation runs full-volume sliding-window inference, much faster on GPU).
4. **Settings -> Internet** -> On.

## 1. Clone repo and install dependencies

In [ ]:
import os

# Idempotent: safe to re-run this cell without restarting the session -
# re-running `git clone` after %cd already moved into the repo would nest
# a second clone inside the first instead of updating it.
if os.path.basename(os.getcwd()) != "brain_tumor_segmentation":
    if not os.path.isdir("brain_tumor_segmentation"):
        !git clone https://github.com/thatavidreader/brain_tumor_segmentation.git
    %cd brain_tumor_segmentation
else:
    !git pull

!pip install -q -r requirements.txt

## 2. Locate the attached dataset and checkpoint

In [ ]:
import glob
import os

data_matches = glob.glob("/kaggle/input/**/BraTS20_Training_001", recursive=True)
if not data_matches:
    raise FileNotFoundError("BraTS dataset not found under /kaggle/input - attach it via Add Data.")
data_dir = os.path.dirname(data_matches[0])
print("Dataset dir:", data_dir)

ckpt_matches = glob.glob("/kaggle/input/**/best_model.pth", recursive=True)
if not ckpt_matches:
    raise FileNotFoundError("best_model.pth not found under /kaggle/input - upload it via Add Data -> Upload.")
checkpoint_path = ckpt_matches[0]
print("Checkpoint:", checkpoint_path)

## 3. Point config at the attached data
`cache_rate` is forced to `0` below since `get_dataloaders()` builds train/val/test loaders together internally even though evaluation only uses the test one — leaving the default `0.5` would try to cache full-res train/val volumes into RAM, the same OOM crash we hit during training.

In [ ]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

config["paths"]["data_dir"] = data_dir
# get_dataloaders() always builds train/val/test loaders together, even though
# evaluation only uses the test one - without this, the default cache_rate (0.5)
# caches full-res train/val volumes into RAM and OOM-crashes the kernel.
config["data"]["cache_rate"] = 0.0

with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f)

## 4. Run test-set evaluation
Regenerates the same train/val/test split as training (same seed + same case list -> identical split, no need for the old `splits.json`). Reports Dice/IoU/HD95 per class and saves 4 GT-vs-prediction comparison figures.

In [ ]:
!python -m src.evaluate --config config.yaml --checkpoint "$checkpoint_path"

## 5. View evaluation comparison figures

In [ ]:
import json
from IPython.display import Image, display

with open("outputs/evaluation/test_metrics.json") as f:
    results = json.load(f)
print(json.dumps(results["overall"], indent=2))

for fig_path in sorted(glob.glob("outputs/visualizations/eval_*.png")):
    print(fig_path)
    display(Image(filename=fig_path))

## 6. Single-case inference on one test case
Picks the first case from the test split (written by step 4 to `data/splits.json`) and runs full inference + a GT-vs-prediction figure.

In [ ]:
with open("data/splits.json") as f:
    splits = json.load(f)

test_case_id = splits["test"][0]
case_dir = os.path.join(data_dir, test_case_id)
print("Running inference on:", test_case_id)

!python inference.py --config config.yaml --checkpoint "$checkpoint_path" --case-dir "$case_dir" --case-id "$test_case_id" --out-dir outputs/predictions

In [ ]:
comparison_png = f"outputs/predictions/{test_case_id}_comparison.png"
display(Image(filename=comparison_png))

## 7. Save Version
Click **Save Version -> "Save & Run All (Commit)"** so `outputs/evaluation/test_metrics.json`, the comparison figures, and the predicted NIfTI persist as this notebook's downloadable Output.